# Complete MoE PM2.5 forecasting workflow with reviewer revisions

**Run this notebook from top to bottom.** It contains the original research workflow and the reviewer additions in one sequence, starting with raw data preparation and ending with manuscript tables, figures and statistical comparisons. You do not need to run cells from the older notebook.

## Before starting
- In Colab select **Runtime → Change runtime type → GPU**.
- Edit paths and settings in **Step 2 only** for a normal run. The existing merged hourly CSV is accepted; alternatively, supply the 2022 CSV and 2023/2024 ZIPs used in the original workflow.
- The merged file must contain hourly **observations before imputation**. If missing PM2.5 has already been replaced by zero, rebuild from source files; this notebook cannot identify which zeros were originally missing.
- `MODE='full'` runs the complete study: **19 configurations × 3 seeds = 57 model/seed experiments**, plus two heuristic baselines. The transfer model has two training phases. An optional `smoke` setting provides a short execution check but does not produce publishable results.
- Completed experiment checkpoints are reused when the data/settings signature matches. An unfinished training run restarts; completed experiments are skipped.

## Execution sequence

| Steps | Work |
|---|---|
| 1–4 | Setup, one settings section, raw ZIP/CSV cleaning, hourly aggregation |
| 5–7 | Experiment plan, chronological splits, training-only preprocessing, data figures |
| 8–10 | Model definitions, execution checks, all baseline and reviewer experiments |
| 11–12 | Forecast/gate/residual/learning figures, station map, comparative statistics |
| 13–14 | Optional filling sensitivity, final results ZIP |

**Checks completed during preparation:** syntax, synthetic raw ingestion, quality-count handling, target preservation, split isolation, metrics, Holm correction, bootstrap logic, diagnostic figures and packaging. **The actual dataset and TensorFlow training were not run here.** Model execution checks are included in Step 9. All research results are generated by your run; none are hard-coded.

# Peak-Weighted Mixture-of-Experts for Multi-Station PM2.5 Forecasting

This notebook is the cleaned public-release workflow associated with the revised manuscript. It contains the preprocessing, chronological split construction, model definitions, reviewer-requested ablations, multi-seed training, evaluation, dependence-aware statistical comparisons, and result packaging.

**Important:** The observational data are not redistributed in this repository. Obtain the CUPI-G data from the original source described in the manuscript and set `DATA_PATH` / `YEAR_SOURCES` in Step 2.

**Reproducibility:** Run Steps 1–14 in order. The notebook intentionally contains no cached execution outputs. The public version does not include exploratory follow-up cells that are not part of the reported benchmark.


## Step 1 — Connect storage and check the runtime

Mount Google Drive to retain results across Colab sessions. For a local runtime, set local paths in Step 2.

In [ ]:
try:
    from google.colab import drive
except ImportError:
    print('Local runtime: configure local paths in Step 2.')
else:
    drive.mount('/content/drive')
# Colab normally includes these libraries. If an import is missing, run:
# %pip install tensorflow pandas numpy scipy matplotlib

## Step 2 — Set all file paths and experiment settings

`INPUT_MODE='auto'` uses `DATA_PATH` when that CSV exists, otherwise it builds the hourly table from `YEAR_SOURCES`. Each year source may be a CSV, ZIP or directory. Change the paths to your actual uploaded files or Drive locations.

Use `STATION_ALLOWLIST` if a source includes non-Delhi stations. `EXPECTED_STATIONS=47` checks the count, not geographic identity. `STATION_METADATA` is optional for training but is needed for the reviewer-requested map; its columns are `station_id, latitude, longitude`.

The threshold 150 µg/m³ is treated as a **study-defined high-pollution threshold**. Verify the manuscript's AQI attribution separately. `full` completes the computational suite; `core` runs only the six principal ablations; `smoke` runs two one-epoch checks.

In [ ]:
from pathlib import Path
import os, json, hashlib, time, gc, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

DATA_PATH = Path('/content/cupig_allstations_hourly_clean_2022_2024.csv')
INPUT_MODE = 'auto'  # existing DATA_PATH first; otherwise convert YEAR_SOURCES below
YEAR_SOURCES = {
    2022: Path('/content/cupig_allstations_hourly_clean (1).csv'),
    2023: Path('/content/1H_20230901_20231130.zip'),
    2024: Path('/content/1H_20240901_20241130.zip'),
}
MIN_VALID_TEN_MINUTES = 4  # Minimum accepted observations out of six per hour
TL_LR = 5e-4
OUT_ROOT = Path('/content/drive/MyDrive/MoE_revision')  # Mount Drive first, or change to /content/MoE_revision
STATION_METADATA = None  # Optional CSV: station_id, latitude, longitude (verified coordinates only)
STATION_ALLOWLIST = None  # Optional list of the verified Delhi station IDs
EXPECTED_STATIONS = 47   # Change only after checking the geographic inclusion criteria
MODE = 'full'          # 'smoke': execution check; 'core': primary ablations; 'full': all reviewer experiments
SEEDS = [1, 7, 21]
L, H = 72, 24
THRESHOLD = 150.0       # Study-defined high-pollution threshold; not asserted to be an AQI category
EPOCHS, BATCH, LR = 30, 256, 1e-3
PATIENCE, EMB_DIM, HIDDEN, DROPOUT = 4, 12, 128, 0.2
FILL_HOURS = 6
BOOTSTRAPS, BLOCK_DAYS = 1000, [3, 7, 14]
TZ = 'Asia/Kolkata'
FEATURES = ['pm25','RH_high','CO_mean','delta_CO_mean','CO_missing','dCO_missing',
            'hod_sin','hod_cos','dow_sin','dow_cos']


RUN_MODEL_CHECKS = True
RUN_FILL_SENSITIVITY = False

## Step 3 — Load shared utilities

These helpers provide file fingerprints, timestamp conversion and descriptive statistics. No preprocessing statistics are estimated yet.

In [ ]:
def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))

def parse_ist(values):
    ts = pd.to_datetime(values, errors='coerce')
    if ts.dt.tz is None: return ts.dt.tz_localize(TZ)
    return ts.dt.tz_convert(TZ)

def describe_values(values):
    a = np.asarray(values, dtype=float).ravel(); a = a[np.isfinite(a)]
    if not len(a): return dict(n=0)
    return dict(n=len(a), mean=a.mean(), std=a.std(ddof=1) if len(a)>1 else np.nan,
                minimum=a.min(), maximum=a.max(), median=np.median(a),
                p90=np.quantile(a,.90), p95=np.quantile(a,.95), p99=np.quantile(a,.99),
                high_pollution_pct=100*np.mean(a>=THRESHOLD))

## Step 4 — Read raw files and construct hourly observations

This restores the original ingestion stage: two-row CUPI-G headers, missing sentinel values, station IDs, IST timestamps, ten-minute to hourly aggregation and yearly merging. Ten-minute inputs require at least four accepted values per hour; already-hourly source files are retained with explicit zero-count values marked missing. RH flags preserve unavailable humidity as missing.

Errors and overlapping timestamps are reported rather than silently omitted. The conversion writes provenance and processing logs. If a valid unimputed merged CSV already exists, it is used directly. Delta CO retains its source definition; it is not recomputed as an hourly difference.

In [ ]:
def normalize_column(name):
    import re
    return re.sub(r'[^a-z0-9]+','',str(name).lower().replace('δ','delta').replace('∆','delta'))

def flatten_cupig_header(frame):
    # CUPI-G exports can have a second header row with mean/std/N.
    if frame.empty: return frame
    first=frame.iloc[0].astype(str).str.strip().str.lower()
    if not first.isin(['mean','std','n']).any(): return frame
    names=[];base=None
    for col,stat in zip(frame.columns,first):
        if normalize_column(col) in ['date','dateutc','hourendist','hourstartist','stationid']:
            names.append(col);base=None;continue
        if not str(col).startswith('Unnamed'): base=str(col).strip()
        names.append(f'{base}_{stat}' if base and stat in ['mean','std','n'] else str(col))
    frame=frame.iloc[1:].copy();frame.columns=names
    if frame.columns.duplicated().any(): raise ValueError('Duplicate columns after flattening two-row header.')
    return frame

def clean_cupig_file(path,year):
    import re
    frame=flatten_cupig_header(pd.read_csv(path,dtype=str))
    aliases={
      'pm25':['pm25','pm25mean'], 'pm25_n':['pm25n'], 'pm25_std':['pm25std'],
      'CO_mean':['comean','co'], 'CO_n':['con'],
      'delta_CO_mean':['deltacomean','deltaco','dcomean','dco'], 'delta_CO_n':['deltacon','dcon'],
      'RH_mean':['humiditymean','humidity','rhmean','rh','relativehumiditymean'],
      'RH_high':['rhhigh'], 'station_id':['stationid']}
    normalized={normalize_column(c):c for c in frame.columns}
    mapping={}
    for canonical,choices in aliases.items():
        match=next((normalized[c] for c in choices if c in normalized),None)
        if match is not None:mapping[match]=canonical
    frame=frame.rename(columns=mapping)
    normalized={normalize_column(c):c for c in frame.columns}
    tscol=next((normalized[c] for c in ['hourendist','date','datetime','timestamp','dateutc'] if c in normalized),None)
    if tscol is None:raise ValueError(f'No timestamp column in {path.name}')
    if normalize_column(tscol)=='dateutc':
        ts=pd.to_datetime(frame[tscol],errors='coerce',utc=True).dt.tz_convert(TZ)
    else:ts=parse_ist(frame[tscol])
    raw_count=len(frame);invalid=int(ts.isna().sum());frame['time']=ts
    frame=frame.dropna(subset=['time'])
    wrong_year=int((frame.time.dt.year!=year).sum())
    if wrong_year:raise ValueError(f'{path.name}: {wrong_year} rows outside declared source year {year}.')
    if 'station_id' not in frame:
        sid=re.sub(r'^CUPIG_','',path.stem,flags=re.I)
        frame['station_id']=sid
    if frame.station_id.isna().any():raise ValueError(f'{path.name}: missing station IDs.')
    frame['station_id']=frame.station_id.astype(str)
    if 'pm25' not in frame:raise ValueError(f'{path.name}: PM2.5 column not recognized. Columns: {list(frame)}')
    for c in ['pm25','pm25_n','pm25_std','CO_mean','CO_n','delta_CO_mean','delta_CO_n','RH_mean','RH_high']:
        if c in frame:frame[c]=pd.to_numeric(frame[c],errors='coerce').replace([-99999,np.inf,-np.inf],np.nan)
    frame.loc[frame.pm25<0,'pm25']=np.nan
    for c,n in [('pm25','pm25_n'),('CO_mean','CO_n'),('delta_CO_mean','delta_CO_n')]:
        if c not in frame:frame[c]=np.nan
        if n in frame:frame.loc[frame[n]<=0,c]=np.nan
    outputs=[];logs=[]
    for sid,g in frame.groupby('station_id',sort=True):
        g=g.sort_values('time')
        if g.time.duplicated().any():raise ValueError(f'{path.name}: duplicate timestamps at {sid}.')
        diffs=g.time.diff().dropna()
        cadence=diffs.mode().iloc[0] if len(diffs) else pd.Timedelta(hours=1)
        if cadence<pd.Timedelta(hours=1):
            # Accept nominal ten-minute input; reject other resolutions until explicitly configured.
            if cadence!=pd.Timedelta(minutes=10):raise ValueError(f'{path.name}: unexpected cadence {cadence}.')
            a=g.set_index('time')
            cols=[c for c in ['pm25','CO_mean','delta_CO_mean','RH_mean'] if c in a]
            mean=a[cols].resample('1h',closed='right',label='right').mean()
            count=a[cols].resample('1h',closed='right',label='right').count()
            for c in cols:mean.loc[count[c]<MIN_VALID_TEN_MINUTES,c]=np.nan
            mean['pm25_std']=a.pm25.resample('1h',closed='right',label='right').std()
            for c,n in [('pm25','pm25_n'),('CO_mean','CO_n'),('delta_CO_mean','delta_CO_n')]:mean[n]=count[c]
            if 'RH_mean' not in mean and 'RH_high' in a:
                # Already-derived flags cannot reconstruct hourly RH. Preserve unavailable values explicitly.
                mean['RH_high']=a.RH_high.resample('1h',closed='right',label='right').mean()
                mean.loc[~mean.RH_high.isin([0,1]),'RH_high']=np.nan
            out=mean.reset_index()
        else:
            if ((g.time.dt.minute!=0)|(g.time.dt.second!=0)).any():raise ValueError(f'{path.name}: hourly data are not end-of-hour aligned.')
            cols=[c for c in ['time','pm25','pm25_n','pm25_std','CO_mean','CO_n','delta_CO_mean','delta_CO_n','RH_mean','RH_high'] if c in g]
            out=g[cols].copy()
        if 'RH_mean' in out:
            out['RH_high']=np.where(out.RH_mean.isna(),np.nan,(out.RH_mean>70).astype(float))
        elif 'RH_high' not in out:out['RH_high']=np.nan
        out['station_id']=sid;out['hour_end_ist']=out.time
        out['dateUTC']=out.time.dt.tz_convert('UTC').dt.strftime('%Y-%m-%d %H:%M:%S')
        out['campaign_year']=year
        out=out.drop(columns='time')
        outputs.append(out)
        logs.append(dict(source=str(path),year=year,station_id=sid,input_rows=len(g),
                         raw_file_rows=raw_count,invalid_timestamp_rows_in_file=invalid,
                         source_cadence=str(cadence),hourly_rows=len(out),
                         valid_pm25_hours=int(out.pm25.notna().sum())))
    if not outputs:raise ValueError(f'No valid records in {path}')
    return pd.concat(outputs,ignore_index=True),logs

def source_csvs(source,extract_dir):
    import zipfile
    source=Path(source)
    if source.is_dir():return sorted(source.rglob('*.csv'))
    if source.suffix.lower()=='.csv':return [source]
    if source.suffix.lower()!='.zip':raise ValueError(f'Expected CSV, directory or ZIP: {source}')
    extract_dir=Path(extract_dir);extract_dir.mkdir(parents=True,exist_ok=True)
    selected=[]
    with zipfile.ZipFile(source) as archive:
        for info in archive.infolist():
            if info.is_dir() or not info.filename.lower().endswith('.csv'):continue
            rel=Path(info.filename)
            if '__MACOSX' in rel.parts:continue
            if rel.is_absolute() or '..' in rel.parts:raise ValueError('Unsafe ZIP member path.')
            dest=extract_dir/rel;dest.parent.mkdir(parents=True,exist_ok=True)
            with archive.open(info) as a,open(dest,'wb') as b:
                import shutil
                shutil.copyfileobj(a,b)
            selected.append(dest)
    return sorted(selected)

def get_hourly_data():
    # auto: use the configured merged CSV if present; otherwise construct it from year sources.
    if INPUT_MODE not in ['auto','merged_hourly','year_sources']:raise ValueError('Invalid INPUT_MODE')
    if INPUT_MODE=='merged_hourly' or (INPUT_MODE=='auto' and DATA_PATH.exists()):
        if not DATA_PATH.exists():raise FileNotFoundError(f'Set DATA_PATH to your merged hourly CSV: {DATA_PATH}')
        print('Using existing hourly observations:',DATA_PATH)
        return DATA_PATH
    folder=OUT_ROOT/'hourly_preparation';folder.mkdir(parents=True,exist_ok=True)
    missing=[str(p) for p in YEAR_SOURCES.values() if not Path(p).exists()]
    if missing:raise FileNotFoundError('Set YEAR_SOURCES to your actual files, or use an existing DATA_PATH. Missing: '+str(missing))
    frames=[];logs=[];failures=[];source_manifest=[]
    for year,source in YEAR_SOURCES.items():
        source=Path(source)
        source_manifest.append(dict(year=year,path=str(source),sha256=file_hash(source) if source.is_file() else 'directory; individual files hashed below'))
        files=source_csvs(source,folder/f'extracted_{year}')
        if not files:raise ValueError(f'No CSV files in {source}')
        for f in files:
            try:
                cleaned,report=clean_cupig_file(f,int(year));frames.append(cleaned);logs.extend(report)
                source_manifest.append(dict(year=year,path=str(f),sha256=file_hash(f)))
            except Exception as exc:failures.append(dict(file=str(f),error=str(exc)))
    pd.DataFrame(logs).to_csv(folder/'raw_processing_log.csv',index=False)
    save_json(folder/'source_manifest.json',source_manifest)
    if failures:
        pd.DataFrame(failures).to_csv(folder/'raw_processing_errors.csv',index=False)
        raise ValueError('Raw conversion encountered errors. See raw_processing_errors.csv; no files were silently omitted.')
    combined=pd.concat(frames,ignore_index=True,sort=False)
    combined['hour_end_ist']=parse_ist(combined.hour_end_ist)
    dup=combined.duplicated(['station_id','hour_end_ist'],keep=False)
    if dup.any():
        combined[dup].to_csv(folder/'duplicate_source_hours.csv',index=False)
        raise ValueError('Overlapping source records; resolve duplicate_source_hours.csv before continuing.')
    combined=combined.sort_values(['station_id','hour_end_ist'])
    dest=folder/'cupig_allstations_hourly_clean_2022_2024.csv';combined.to_csv(dest,index=False)
    print('Hourly observations saved:',dest,'Rows:',len(combined),'Stations:',combined.station_id.nunique())
    return dest

DATA_PATH = get_hourly_data()

## Step 5 — Define the full experiment plan

The original trained model families are retained: LSTM, GRU, BiConvLSTM, TCN, Transformer–Huber, probabilistic LSTM, transfer-learning LSTM with attention, and MoE. Persistence and lag-24 seasonal naive are added during evaluation.

Reviewer controls include a single expert with the same weighted loss, the unweighted counterparts, a fixed equal gate, no station embedding, a three-expert model, a wider single expert, and weights 2×, 4×, 5×, 10× alongside 1× and 3×. Parameter counts are reported; the wider single expert is only approximately capacity comparable.

In [ ]:
def experiment_specs(mode):
    def spec(name,k=2,w=3.,embed=True,gate='learned',kind='lstm',width=96):
        return dict(name=name,k=k,weight=w,embed=embed,gate=gate,kind=kind,width=width)
    core=[spec('moe2_w3'),spec('single_w3',k=1),spec('single_w1',k=1,w=1),
          spec('moe2_w1',w=1),spec('moe2_fixed_gate',gate='fixed'),
          spec('moe2_no_station',embed=False)]
    if mode=='smoke': return core[:2]
    if mode=='core': return core
    if mode!='full': raise ValueError(mode)
    return core+[spec(f'moe2_w{w}',w=w) for w in [2,4,5,10]]+[
        spec('moe3_w3',k=3),spec('single_w3_capacity',k=1,width=192),
        spec('gru',k=1,w=1,embed=False,kind='gru'),
        spec('plain_lstm',k=1,w=1,embed=False),
        spec('biconvlstm',k=1,w=1,embed=False,kind='biconvlstm'),
        spec('tcn',k=1,w=1,embed=False,kind='tcn'),
        spec('transformer_huber',k=1,w=1,embed=False,kind='transformer'),
        spec('prob_lstm',k=1,w=1,kind='prob'),
        spec('tl_lstm_mha',k=1,w=1,embed=False,kind='tl_lstm_mha')]

print(pd.DataFrame(experiment_specs(MODE)).to_string(index=False))

## Step 6 — Prepare observed targets and chronological splits

**Reviewer issues addressed: missingness, leakage, data screening, train/test selection and target transformation.**

Windows require 72 observed PM2.5 history hours and 24 observed future targets. Covariates use causal forward filling over contiguous hourly records, then training-only station/global fallback values. Missing targets are never replaced by zero. All 96 hours of a window must lie in one partition; windows crossing split boundaries are discarded. The last-10% validation cutoff comes from pre-2024 candidate times. All feature scaling and log-target normalization use training data only.

Coverage and missingness tables distinguish recorded hours from absent timestamps. Unique station-hour statistics and repeated forecast-target statistics are reported separately. The actual coverage may be seasonal campaigns; do not automatically describe it as continuous January–December data.

All models now share standardized log targets. Some older baseline cells used raw targets, so regenerate every result and document the revised common protocol. Corrected counts need not equal the previously reported 181,843 windows.

In [ ]:
def prepare_data(
    path,
    out,
    fill_hours=6,
    expected_stations=EXPECTED_STATIONS,
):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)

    # =========================================================
    # 1. Read input data
    # =========================================================
    d = pd.read_csv(
        path,
        dtype={"station_id": str},
    )

    required = {
        "station_id",
        "hour_end_ist",
        "pm25",
        "CO_mean",
        "delta_CO_mean",
        "RH_high",
    }

    missing_columns = required - set(d.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    counts = {"input_rows": int(len(d))}

    d["time"] = parse_ist(d["hour_end_ist"])

    counts["invalid_time_or_station_rows"] = int(
        (
            d["time"].isna()
            | d["station_id"].isna()
        ).sum()
    )

    d = d.dropna(
        subset=["time", "station_id"]
    ).copy()

    d = d[
        d["time"].dt.year.isin([2022, 2023, 2024])
    ].copy()

    if STATION_ALLOWLIST is not None:
        d = d[
            d["station_id"].isin(STATION_ALLOWLIST)
        ].copy()

    # =========================================================
    # 2. Explicit station exclusions
    # =========================================================
    exclusion_reasons = {
        "NS-31": (
            "No records before the original training cutoff"
        ),
        "NS-22": (
            "No complete training windows in the preceding run"
        ),
        "NS-23": (
            "No complete training windows in the preceding run"
        ),
    }

    excluded_station_ids = set(exclusion_reasons)

    station_count_before = int(
        d["station_id"].nunique()
    )

    excluded = d[
        d["station_id"].isin(excluded_station_ids)
    ].copy()

    removed_station_count = int(
        excluded["station_id"].nunique()
    )

    exclusion_report = (
        excluded.groupby("station_id")
        .agg(
            first_record=("time", "min"),
            last_record=("time", "max"),
            recorded_rows=("time", "size"),
        )
        .reset_index()
    )

    exclusion_report["reason"] = (
        exclusion_report["station_id"]
        .map(exclusion_reasons)
    )

    exclusion_report.to_csv(
        out / "excluded_stations.csv",
        index=False,
    )

    if not excluded.empty:
        print("\nExcluded from the forecasting cohort:")
        print(
            exclusion_report.to_string(index=False)
        )

    counts["requested_station_exclusions"] = sorted(
        excluded_station_ids
    )

    counts["excluded_station_ids"] = sorted(
        excluded["station_id"].unique().tolist()
    )

    counts["excluded_station_rows"] = int(
        len(excluded)
    )

    counts["stations_before_exclusion"] = (
        station_count_before
    )

    counts["expected_stations_before_exclusion"] = (
        int(expected_stations)
        if expected_stations is not None
        else None
    )

    d = d[
        ~d["station_id"].isin(excluded_station_ids)
    ].copy().reset_index(drop=True)

    # Adjust only when the configured expectation matched
    # the observed original cohort.
    if (
        expected_stations is not None
        and expected_stations == station_count_before
    ):
        expected_stations -= removed_station_count

    counts["expected_stations_after_exclusion"] = (
        int(expected_stations)
        if expected_stations is not None
        else None
    )

    if d.empty:
        raise ValueError(
            "No records remain after year and station filtering."
        )

    # =========================================================
    # 3. Validate timestamps and observation values
    # =========================================================
    duplicate_mask = d.duplicated(
        ["station_id", "time"],
        keep=False,
    )

    if duplicate_mask.any():
        d.loc[duplicate_mask].to_csv(
            out / "duplicate_rows.csv",
            index=False,
        )

        raise ValueError(
            "Duplicate station timestamps found. "
            "Resolve using source provenance; "
            "see duplicate_rows.csv."
        )

    non_hourly = (
        (d["time"].dt.minute != 0)
        | (d["time"].dt.second != 0)
        | (d["time"].dt.microsecond != 0)
        | (d["time"].dt.nanosecond != 0)
    )

    if non_hourly.any():
        raise ValueError(
            "Expected end-of-hour timestamps. "
            "Confirm aggregation before running."
        )

    for c in [
        "pm25",
        "CO_mean",
        "delta_CO_mean",
        "RH_high",
    ]:
        d[c] = pd.to_numeric(
            d[c],
            errors="coerce",
        ).replace(
            [-99999, np.inf, -np.inf],
            np.nan,
        )

    counts["negative_pm25_rows"] = int(
        (d["pm25"] < 0).sum()
    )

    d.loc[d["pm25"] < 0, "pm25"] = np.nan

    for c, count_column in [
        ("pm25", "pm25_n"),
        ("CO_mean", "CO_n"),
        ("delta_CO_mean", "delta_CO_n"),
    ]:
        if count_column in d.columns:
            invalid_count = (
                pd.to_numeric(
                    d[count_column],
                    errors="coerce",
                ) <= 0
            )

            d.loc[invalid_count, c] = np.nan

    if not d["RH_high"].dropna().isin([0, 1]).all():
        raise ValueError(
            "RH_high must be 0/1 or missing."
        )

    d = d.sort_values(
        ["station_id", "time"]
    ).reset_index(drop=True)

    # =========================================================
    # 4. Construct complete observed forecasting windows
    # =========================================================
    origins = []
    coverage = []

    for sid, g in d.groupby(
        "station_id",
        sort=True,
    ):
        for year, q in g.groupby(
            g["time"].dt.year
        ):
            expected_hours = int(
                (
                    q["time"].max()
                    - q["time"].min()
                ) / pd.Timedelta(hours=1)
            ) + 1

            coverage.append(
                {
                    "station_id": sid,
                    "year": int(year),
                    "first": q["time"].min(),
                    "last": q["time"].max(),
                    "recorded_hours": int(len(q)),
                    "within_year_span_hours": expected_hours,
                    "absent_timestamp_hours": (
                        expected_hours - len(q)
                    ),
                }
            )

        # Each segment contains consecutive hourly records.
        segment = (
            g["time"].diff()
            != pd.Timedelta(hours=1)
        ).cumsum()

        for _, q in g.groupby(segment):
            if len(q) < L + H:
                continue

            observed = np.isfinite(
                q["pm25"].to_numpy()
            )

            invalid = np.r_[
                0,
                np.cumsum(~observed),
            ]

            starts = np.arange(
                len(q) - L - H + 1
            )

            complete_window = (
                invalid[starts + L + H]
                - invalid[starts]
            ) == 0

            starts = starts[complete_window]

            origins.extend(
                q.index.to_numpy()[
                    starts + L - 1
                ]
            )

    pd.DataFrame(coverage).to_csv(
        out / "station_coverage.csv",
        index=False,
    )

    origins = np.asarray(
        origins,
        dtype=np.int64,
    )

    if not len(origins):
        raise ValueError(
            f"No complete {L}+{H} hour windows. "
            "Check station_coverage.csv and missing PM2.5."
        )

    # =========================================================
    # 5. Chronological train, validation and test splits
    # =========================================================
    times = d["time"].array

    test_start = pd.Timestamp(
        "2024-01-01",
        tz=TZ,
    )

    test_end = pd.Timestamp(
        "2025-01-01",
        tz=TZ,
    )

    pre = origins[
        times[origins + H] < test_start
    ]

    if not len(pre):
        raise ValueError(
            "No complete pre-2024 windows."
        )

    # The cutoff uses only pre-test candidate times.
    ordered = np.sort(
        times[pre].asi8
    )

    cut = pd.Timestamp(
        ordered[int(0.9 * len(ordered))],
        tz="UTC",
    ).tz_convert(TZ)

    # Full-window purging:
    # histories and targets cannot cross split boundaries.
    mask = {
        "train": np.asarray(
            times[origins + H] < cut
        ),
        "val": np.asarray(
            (
                times[origins - L + 1] >= cut
            )
            & (
                times[origins + H] < test_start
            )
        ),
        "test": np.asarray(
            (
                times[origins - L + 1] >= test_start
            )
            & (
                times[origins + H] < test_end
            )
        ),
    }

    print("\nTraining cutoff:", cut)

    # Save window counts before the eligibility checks.
    window_counts = pd.DataFrame(
        {
            "station_id": sorted(
                d["station_id"].unique()
            )
        }
    ).set_index("station_id")

    for name, split_mask in mask.items():
        split_origins = origins[split_mask]

        split_counts = (
            d.loc[split_origins, "station_id"]
            .value_counts()
        )

        window_counts[name + "_windows"] = (
            split_counts.reindex(
                window_counts.index,
                fill_value=0,
            ).astype("int64")
        )

    window_counts.to_csv(
        out / "station_window_counts.csv"
    )

    stations = sorted(
        d.loc[
            d["time"] < cut,
            "station_id",
        ].unique()
    )

    unknown = sorted(
        set(d["station_id"])
        - set(stations)
    )

    if unknown:
        raise ValueError(
            f"Stations absent from training: {unknown}. "
            "Check station_coverage.csv and "
            "station_window_counts.csv. "
            "A separate transfer protocol or an explicit "
            "cohort decision is required."
        )

    if (
        expected_stations is not None
        and len(stations) != expected_stations
    ):
        raise ValueError(
            f"Found {len(stations)} stations, "
            f"expected {expected_stations}. "
            "Verify the station inclusion list and "
            "excluded_stations.csv."
        )

    # Check actual training windows before preprocessing.
    train_station_ids = set(
        d.loc[
            origins[mask["train"]],
            "station_id",
        ]
    )

    for part in ["val", "test"]:
        evaluation_station_ids = set(
            d.loc[
                origins[mask[part]],
                "station_id",
            ]
        )

        unseen = sorted(
            evaluation_station_ids
            - train_station_ids
        )

        if unseen:
            raise ValueError(
                f"{part} has stations without complete "
                f"training windows: {unseen}. "
                "See station_window_counts.csv. "
                "Do not bypass this check."
            )

    for name, split_mask in mask.items():
        if not split_mask.any():
            raise ValueError(
                f"Empty {name} split after purging."
            )

    station_mapping = {
        sid: i
        for i, sid in enumerate(stations)
    }

    d["station_idx"] = (
        d["station_id"]
        .map(station_mapping)
        .astype("int32")
    )

    d["split"] = np.where(
        d["time"] < cut,
        "train",
        np.where(
            d["time"] < test_start,
            "val",
            "test",
        ),
    )

    d["CO_missing"] = (
        d["CO_mean"]
        .isna()
        .astype("float32")
    )

    d["dCO_missing"] = (
        d["delta_CO_mean"]
        .isna()
        .astype("float32")
    )

    raw = d.copy()

    # =========================================================
    # 6. Training-only covariate imputation
    # =========================================================
    audit = []
    fitted = {}

    for c in [
        "CO_mean",
        "delta_CO_mean",
        "RH_high",
    ]:
        train = raw[
            raw["split"] == "train"
        ]

        if c == "RH_high":
            med = train.groupby(
                "station_id"
            )[c].agg(
                lambda values: (
                    values.mode().iloc[0]
                    if len(values.mode())
                    else np.nan
                )
            )

            modes = train[c].mode()

            fallback = (
                float(modes.iloc[0])
                if len(modes)
                else 0.0
            )

        else:
            med = train.groupby(
                "station_id"
            )[c].median()

            fallback = train[c].median()

            if not np.isfinite(fallback):
                fallback = 0.0

        fitted[c] = {
            "station_values": med.to_dict(),
            "global_fallback": float(fallback),
        }

        for sid, g in d.groupby("station_id"):
            segment = (
                g["time"].diff()
                != pd.Timedelta(hours=1)
            ).cumsum()

            filled = g[c].copy()

            # Only earlier observations are used.
            # Forward filling cannot cross timestamp gaps.
            if fill_hours:
                filled = (
                    g[c]
                    .groupby(segment)
                    .ffill(limit=fill_hours)
                )

            original = g[c].notna()

            forward = (
                ~original
                & filled.notna()
            )

            value = med.get(
                sid,
                np.nan,
            )

            use_global = not np.isfinite(value)

            if use_global:
                value = fallback

            for part, idx in (
                g.groupby("split").groups.items()
            ):
                missing = ~original.loc[idx]
                remaining = filled.loc[idx].isna()

                audit.append(
                    {
                        "station_id": sid,
                        "split": part,
                        "variable": c,
                        "recorded_hours": int(len(idx)),
                        "missing_hours": int(
                            missing.sum()
                        ),
                        "missing_pct": float(
                            100 * missing.mean()
                        ),
                        "observed_hours": int(
                            original.loc[idx].sum()
                        ),
                        "forward_filled_hours": int(
                            forward.loc[idx].sum()
                        ),
                        "station_fill_hours": (
                            int(remaining.sum())
                            if not use_global
                            else 0
                        ),
                        "global_fallback_hours": (
                            int(remaining.sum())
                            if use_global
                            else 0
                        ),
                    }
                )

            d.loc[g.index, c] = (
                filled.fillna(value)
            )

    # =========================================================
    # 7. Cyclic temporal features
    # =========================================================
    for period, denominator in [
        ("hod", 24),
        ("dow", 7),
    ]:
        values = (
            d["time"].dt.hour
            if period == "hod"
            else d["time"].dt.dayofweek
        )

        d[period + "_sin"] = np.sin(
            2 * np.pi * values / denominator
        )

        d[period + "_cos"] = np.cos(
            2 * np.pi * values / denominator
        )

    # =========================================================
    # 8. Fit input scaling on training rows only
    # =========================================================
    xr = d[FEATURES].to_numpy(
        dtype="float32"
    )

    train_rows = np.flatnonzero(
        d["split"].eq("train").to_numpy()
    )

    mu = np.nanmean(
        xr[train_rows],
        axis=0,
    )

    sd = np.nanstd(
        xr[train_rows],
        axis=0,
    )

    sd = np.maximum(
        sd,
        1e-6,
    )

    if not (
        np.isfinite(mu).all()
        and np.isfinite(sd).all()
    ):
        raise ValueError(
            "Non-finite training scaling statistics. "
            "Check whether a feature is entirely missing "
            "in training."
        )

    x = (xr - mu) / sd

    # =========================================================
    # 9. Build target arrays and save manifests
    # =========================================================
    data = {}

    for name, split_mask in mask.items():
        o = origins[split_mask]

        o = o[
            np.argsort(
                times[o].asi8,
                kind="stable",
            )
        ]

        y = raw["pm25"].to_numpy()[
            o[:, None]
            + np.arange(1, H + 1)
        ]

        if not np.isfinite(y).all():
            raise ValueError(
                f"Non-finite targets in the {name} split."
            )

        data[name] = {
            "origin": o,
            "y": y.astype("float32"),
            "sid": (
                d["station_idx"].to_numpy()[o]
            ),
            "time_ns": times[o].asi8,
        }

        np.save(
            out / f"{name}_targets_raw.npy",
            data[name]["y"],
        )

        np.save(
            out / f"{name}_station_indices.npy",
            data[name]["sid"],
        )

        manifest = pd.DataFrame(
            {
                "station_id": (
                    d["station_id"].to_numpy()[o]
                ),
                "origin": times[o],
                "input_start": (
                    times[o - L + 1]
                ),
                "target_start": times[o + 1],
                "target_end": times[o + H],
            }
        )

        manifest.to_csv(
            out / f"{name}_window_manifest.csv",
            index=False,
        )

    # =========================================================
    # 10. Training-only target transformation
    # =========================================================
    yl = np.log1p(
        data["train"]["y"]
    )

    ym = float(yl.mean())

    ys = max(
        float(yl.std()),
        1e-6,
    )

    for part_data in data.values():
        part_data["z"] = (
            (
                np.log1p(part_data["y"])
                - ym
            ) / ys
        ).astype("float32")

    # Validate every actual model input window.
    for name, part_data in data.items():
        chunks = np.array_split(
            part_data["origin"],
            max(
                1,
                len(part_data["origin"]) // 10000,
            ),
        )

        for chunk in chunks:
            window_values = x[
                chunk[:, None]
                + np.arange(-L + 1, 1)
            ]

            if not np.isfinite(window_values).all():
                raise ValueError(
                    f"Non-finite input values in {name} windows."
                )

    # Missing PM2.5 is never filled in usable windows.
    # Only unused values are made finite for tensor storage.
    x = np.nan_to_num(x).astype("float32")

    # =========================================================
    # 11. Missingness reports
    # =========================================================
    ad = pd.DataFrame(audit)

    ad.to_csv(
        out / "missingness_imputation_by_station.csv",
        index=False,
    )

    agg = (
        ad.groupby(["split", "variable"])
        .sum(numeric_only=True)
        .drop(columns="missing_pct")
    )

    agg["missing_pct"] = (
        100
        * agg["missing_hours"]
        / agg["recorded_hours"]
    )

    agg.to_csv(
        out / "missingness_imputation_summary.csv"
    )

    # =========================================================
    # 12. Descriptive statistics
    # =========================================================
    stats = []

    for part, g in raw.groupby("split"):
        stats.append(
            {
                "population": (
                    "unique_recorded_station_hours"
                ),
                "split": part,
                **describe_values(g["pm25"]),
            }
        )

    for part, part_data in data.items():
        stats.append(
            {
                "population": (
                    "forecast_target_positions_with_overlap"
                ),
                "split": part,
                **describe_values(part_data["y"]),
            }
        )

    pd.DataFrame(stats).to_csv(
        out / "pm25_descriptive_statistics.csv",
        index=False,
    )

    monthly = (
        raw.assign(
            month=raw["time"].dt.strftime("%Y-%m")
        )
        .groupby("month")["pm25"]
        .apply(
            lambda values: pd.Series(
                describe_values(values)
            )
        )
    )

    monthly.to_csv(
        out / "monthly_distribution_shift.csv"
    )

    # =========================================================
    # 13. Save audit and preprocessing parameters
    # =========================================================
    assigned_window_mask = np.logical_or.reduce(
        list(mask.values())
    )

    counts.update(
        retained_rows=int(len(d)),
        stations=int(len(stations)),
        candidate_windows=int(len(origins)),
        discarded_boundary_windows=int(
            (~assigned_window_mask).sum()
        ),
        split_windows={
            name: int(len(part_data["origin"]))
            for name, part_data in data.items()
        },
        split_stations={
            name: int(
                len(np.unique(part_data["sid"]))
            )
            for name, part_data in data.items()
        },
        validation_cutoff=str(cut),
        first_record=str(
            raw["time"].min()
        ),
        last_record=str(
            raw["time"].max()
        ),
    )

    save_json(
        out / "data_audit.json",
        counts,
    )

    save_json(
        out / "preprocessing.json",
        {
            "features": FEATURES,
            "x_mean": mu.tolist(),
            "x_std": sd.tolist(),
            "y_log_mean": ym,
            "y_log_std": ys,
            "fit_end_exclusive": str(cut),
            "imputation": fitted,
            "stations": stations,
            "excluded_station_ids": sorted(
                excluded_station_ids
            ),
            "exclusion_reasons": exclusion_reasons,
        },
    )

    print("\nData preparation summary:")
    print(
        json.dumps(
            counts,
            indent=2,
        )
    )

    print("\nStation window counts:")
    print(
        window_counts.to_string()
    )

    print(
        "\nInput schema must originate from observations "
        "BEFORE covariate imputation."
    )

    return {
        "x": x,
        "raw_x": xr,
        "data": data,
        "ym": ym,
        "ys": ys,
        "stations": stations,
        "raw": raw,
        "out": out,
    }


def start_run():
    import tensorflow as tf

    specs = experiment_specs(MODE)

    seeds = (
        [SEEDS[0]]
        if MODE == "smoke"
        else SEEDS
    )

    config = {
        "mode": MODE,
        "seeds": seeds,
        "specs": specs,
        "L": L,
        "H": H,
        "threshold": THRESHOLD,
        "fill_hours": FILL_HOURS,
        "epochs": EPOCHS,
        "batch": BATCH,
        "lr": LR,
        "patience": PATIENCE,
        "hidden": HIDDEN,
        "embedding": EMB_DIM,
        "dropout": DROPOUT,
        "data_sha256": file_hash(DATA_PATH),
        "allowlist": STATION_ALLOWLIST,
        "excluded_station_ids": [
            "NS-31",
            "NS-22",
            "NS-23",
        ],
        "expected_stations_before_exclusion": (
            EXPECTED_STATIONS
        ),
        "protocol": (
            "full_revision_v5_exclude_NS31_NS22_NS23"
        ),
        "tensorflow": tf.__version__,
        "tl_lr": TL_LR,
        "minimum_ten_minute_observations": (
            MIN_VALID_TEN_MINUTES
        ),
    }

    tag = hashlib.sha256(
        json.dumps(
            config,
            sort_keys=True,
        ).encode()
    ).hexdigest()[:12]

    out = OUT_ROOT / (
        MODE + "_" + tag
    )

    out.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("Output folder:", out)

    save_json(
        out / "run_config.json",
        config,
    )

    save_json(
        out / "environment.json",
        {
            "python": platform.python_version(),
            "tensorflow": tf.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "platform": platform.platform(),
            "gpus": [
                str(device)
                for device in (
                    tf.config.list_physical_devices("GPU")
                )
            ],
        },
    )

    try:
        tf.config.experimental.enable_op_determinism()
    except (AttributeError, RuntimeError):
        pass

    pd.DataFrame(specs).to_csv(
        out / "experiment_plan.csv",
        index=False,
    )

    D = prepare_data(
        path=DATA_PATH,
        out=out,
        fill_hours=FILL_HOURS,
        expected_stations=EXPECTED_STATIONS,
    )

    print(
        f"\n{len(specs)} configurations × "
        f"{len(seeds)} seeds = "
        f"{len(specs) * len(seeds)} model/seed experiments "
        "(transfer baseline has two training phases)"
    )

    print("Results:", out)

    return D, specs, seeds


# =============================================================
# Run data preparation
# =============================================================
D, SPECS, ACTIVE_SEEDS = start_run()

print("\nPM2.5 descriptive statistics:")

print(
    pd.read_csv(
        D["out"] / "pm25_descriptive_statistics.csv"
    ).to_string(index=False)
)

## Step 7 — Generate the original data and preprocessing figures

Creates station missingness heatmaps, recorded CO gap-length distribution, monthly high-pollution frequency, target transformation histograms and the 72→24-hour window diagram. Grey heatmap cells indicate unavailable records; the gap histogram only describes consecutive recorded hourly rows.

In [ ]:
def plot_data_diagnostics(D):
    out=D['out'];raw=D['raw'].copy();raw['month']=raw.time.dt.strftime('%Y-%m')
    for c in ['pm25','CO_mean','delta_CO_mean']:
        frame=raw.assign(week=raw.time.dt.tz_localize(None).dt.to_period('W').dt.start_time,
                         missing=raw[c].isna().astype(float))
        grid=frame.groupby(['station_id','week']).missing.mean().unstack().reindex(D['stations'])*100
        fig,ax=plt.subplots(figsize=(12,max(4,len(grid)*.16)))
        cmap=plt.get_cmap('viridis').copy();cmap.set_bad('lightgrey')
        im=ax.imshow(grid.to_numpy(),aspect='auto',vmin=0,vmax=100,cmap=cmap)
        steps=np.linspace(0,len(grid.columns)-1,min(10,len(grid.columns)),dtype=int)
        ax.set_xticks(steps,labels=[str(grid.columns[i].date()) for i in steps],rotation=45,ha='right')
        ax.set_yticks(np.arange(len(grid)),labels=grid.index,fontsize=6)
        ax.set(title=f'{c}: missing fraction of recorded station-hours',xlabel='Week',ylabel='Station')
        fig.colorbar(im,ax=ax,label='Missing (%)');fig.tight_layout();fig.savefig(out/f'missingness_{c}.png',dpi=300);plt.close(fig)
    gaps=[]
    for sid,g in raw.groupby('station_id'):
        breaks=(g.time.diff()!=pd.Timedelta(hours=1)).cumsum()
        for _,q in g.groupby(breaks):
            missing=q.CO_mean.isna().to_numpy();runs=np.diff(np.r_[False,missing,False].astype(int))
            lengths=np.flatnonzero(runs==-1)-np.flatnonzero(runs==1)
            gaps.extend([dict(station_id=sid,gap_hours=int(n)) for n in lengths])
    pd.DataFrame(gaps,columns=['station_id','gap_hours']).to_csv(out/'co_recorded_gap_lengths.csv',index=False)
    if gaps:
        lengths=np.array([r['gap_hours'] for r in gaps]);cap=max(1,int(np.quantile(lengths,.99)))
        fig,ax=plt.subplots(figsize=(8,4));ax.hist(lengths,bins=np.arange(.5,cap+1.5))
        ax.axvline(FILL_HOURS,color='red',ls='--',label=f'Forward fill limit {FILL_HOURS} h')
        ax.set(xlabel='Consecutive recorded hours with CO missing',ylabel='Count',title='CO gaps (x-axis capped at 99th percentile)');ax.legend()
        fig.tight_layout();fig.savefig(out/'co_gap_lengths.png',dpi=300);plt.close(fig)
    valid=raw[raw.pm25.notna()].copy();valid['high']=valid.pm25>=THRESHOLD
    monthly=valid.groupby('month').agg(valid_hours=('pm25','size'),high_hours=('high','sum'))
    monthly['high_pct']=100*monthly.high_hours/monthly.valid_hours
    monthly.to_csv(out/'monthly_high_pollution_frequency.csv')
    fig,ax=plt.subplots(figsize=(11,4));ax.bar(monthly.index,monthly.high_pct)
    ax.tick_params(axis='x',rotation=60);ax.set(ylabel='High-pollution station-hours (%)',title=f'Observed monthly frequency at {THRESHOLD:g} µg/m³')
    fig.tight_layout();fig.savefig(out/'monthly_high_pollution_frequency.png',dpi=300);plt.close(fig)
    y=D['data']['train']['y'].ravel();z=np.log1p(y)
    fig,axes=plt.subplots(1,3,figsize=(12,3.5))
    for ax,a,title in zip(axes,[y,z,(z-D['ym'])/D['ys']],['Observed training targets','Log1p targets','Standardized log1p targets']):
        ax.hist(a,bins=70);ax.set(title=title,ylabel='Target positions')
    fig.tight_layout();fig.savefig(out/'target_transformations.png',dpi=300);plt.close(fig)
    # Precise interval diagram, no artificial pollution data.
    fig,ax=plt.subplots(figsize=(10,3))
    for j,origin in enumerate([0,1,2]):
        ax.broken_barh([(origin-L+1,L)],(j-.3,.6),facecolors='steelblue')
        ax.broken_barh([(origin+1,H)],(j-.3,.6),facecolors='darkorange')
    ax.set(yticks=[0,1,2],yticklabels=['Origin t','Origin t+1','Origin t+2'],xlabel='Relative hour',title='72 observed input hours → 24 future target hours; stride 1 hour')
    ax.axvline(.5,color='black',ls='--');fig.tight_layout();fig.savefig(out/'window_construction.png',dpi=300);plt.close(fig)

plot_data_diagnostics(D)

## Step 8 — Define all neural architectures and losses

**Reviewer issues addressed: complete hyperparameters, loss definition, two versus three experts and component ablations.**

The reference uses the original LSTM(128), station embedding(12), Dense(96) expert heads, dropout 0.2, and a Dense(32) scalar sigmoid gate. The gate is constant over each 24-hour forecast. Every actual architecture is saved as JSON during training.

For each target position, $w_{ih}=1+(\lambda-1)\mathbf1[y_{ih}\ge150]$ and the regression objective is $\operatorname{mean}_{i,h}(w_{ih}|z_{ih}-\hat z_{ih}|)$. No focal loss or other class balancing is used. Expert names are indices; whether one specializes in peaks must be established from diagnostics.

The transfer baseline uses 2022 training windows for pretraining, then freezes its LSTM and fine-tunes with 2023 training windows. Both phases use the same separate validation partition and Nadam at 5e-4. Other settings and any differences from the submitted experiments must be reported accurately.

In [ ]:
def build_model(spec,nstations):
    import tensorflow as tf
    K=tf.keras.layers
    seq=K.Input((L,len(FEATURES)),name='seq_in'); inputs={'seq_in':seq}; x=seq
    if spec['embed']:
        st=K.Input((),dtype='int32',name='station_in'); inputs['station_in']=st
        e=K.Embedding(nstations,EMB_DIM,name='station_embedding')(st)
        x=K.Concatenate()([x,K.RepeatVector(L)(e)])
    kind=spec['kind']
    if kind=='tl_lstm_mha':
        x=K.LSTM(128,return_sequences=True,name='transfer_lstm')(x)
        x=K.Dropout(.3)(x)
        a=K.MultiHeadAttention(num_heads=4,key_dim=32)(x,x)
        a=K.Dropout(.2)(a)
        x=K.LayerNormalization()(K.Concatenate()([x,a]))
        x=K.Dense(64,activation='relu')(x)
        x=K.GlobalAveragePooling1D()(x);x=K.Dropout(.2)(x)
        return tf.keras.Model(inputs,K.Dense(H)(x),name=spec['name'])
    if kind=='biconvlstm':
        x=K.Reshape((L,1,len(FEATURES),1))(x)
        x=K.Bidirectional(K.ConvLSTM2D(32,(1,2),padding='same'))(x)
        x=K.Flatten()(K.LayerNormalization()(x))
    elif kind=='tcn':
        for dilation in [1,2,4,8,16,32]:
            skip=K.Conv1D(64,1)(x)
            for _ in range(2):
                x=K.Conv1D(64,3,padding='causal',dilation_rate=dilation,activation='relu')(x)
                x=K.Dropout(.15)(x)
            x=K.Add()([x,skip])
        x=K.GlobalAveragePooling1D()(x)
    elif kind=='transformer':
        x=K.Dense(64)(x)
        # Fixed sinusoidal positional encoding; explicit, serializable constant addition.
        pos=np.arange(L)[:,None]; freq=np.exp(np.arange(0,64,2)*(-np.log(10000.)/64))
        pe=np.zeros((L,64),'float32'); pe[:,0::2]=np.sin(pos*freq); pe[:,1::2]=np.cos(pos*freq)
        x=x+tf.constant(pe)
        for _ in range(3):
            a=K.MultiHeadAttention(num_heads=4,key_dim=16)(x,x)
            x=K.LayerNormalization()(K.Add()([x,K.Dropout(.1)(a)]))
            f=K.Dense(128,activation='relu')(x);f=K.Dense(64)(K.Dropout(.1)(f))
            x=K.LayerNormalization()(K.Add()([x,f]))
        x=K.GlobalAveragePooling1D()(x)
    else: x=(K.GRU(HIDDEN) if kind=='gru' else K.LSTM(HIDDEN,name='shared_lstm'))(x)
    x=K.Dropout(DROPOUT)(x)
    heads=[]
    for i in range(spec['k']):
        h=K.Dense(spec['width'],activation='relu')(x);h=K.Dropout(DROPOUT)(h)
        h=K.Dense(H*2 if kind=='prob' else H,name=f'expert_{i}')(h)
        heads.append(h)
    if kind=='prob': out=K.Reshape((H,2))(heads[0])
    elif spec['k']==1: out=heads[0]
    elif spec['gate']=='fixed': out=K.Average()(heads)
    else:
        g=K.Dense(32,activation='relu')(x)
        # For two experts, preserve the original single sigmoid gate exactly.
        if spec['k']==2:
            g=K.Dense(1,activation='sigmoid',name='gate')(g)
            out=heads[0]*(1-g)+heads[1]*g
        else:
            g=K.Dense(spec['k'],activation='softmax',name='gate')(g)
            stack=K.Concatenate(axis=1)([K.Reshape((1,H))(h) for h in heads])
            out=K.Dot(axes=(1,1))([g,stack])
    return tf.keras.Model(inputs,out,name=spec['name'])

def loss_for(spec,ym,ys):
    import tensorflow as tf
    thr=(np.log1p(THRESHOLD)-ym)/ys
    if spec['kind']=='prob':
        def nll(y,p):
            mu=p[...,0]; sig=tf.nn.softplus(p[...,1])+1e-6
            return tf.reduce_mean(.5*((y-mu)/sig)**2+tf.math.log(sig)+.5*np.log(2*np.pi))
        return nll
    if spec['kind']=='transformer': return tf.keras.losses.Huber(delta=1.)
    def weighted_mae(y,p):
        w=1+(spec['weight']-1)*tf.cast(y>=thr,tf.float32)
        return tf.reduce_mean(w*tf.abs(y-p))
    return weighted_mae

def plot_architecture(D):
    from matplotlib.patches import FancyBboxPatch
    fig,ax=plt.subplots(figsize=(11,6));ax.set_xlim(0,11);ax.set_ylim(0,6);ax.axis('off')
    boxes={
      'sequence':(0.2,4.3,2.2,.8,'72 × 10 input sequence'),
      'station':(0.2,1.4,2.2,.9,'Station embedding (12)\nrepeated over 72 hours'),
      'encoder':(3,2.7,2.3,1.1,'Concatenate inputs\nLSTM (128)\nDropout 0.2'),
      'expert0':(6.2,4.4,2,.9,'Expert 0\nDense 96 → 24'),
      'gate':(6.2,2.7,2,.9,'Scalar gate\nDense 32 → sigmoid'),
      'expert1':(6.2,1,2,.9,'Expert 1\nDense 96 → 24'),
      'output':(8.8,2.4,2,1.3,'(1 − g) A + g B\n24 forecast steps\nInverse transformation')}
    for key,(x,y,w,h,text) in boxes.items():
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.08',facecolor='#edf3fa',edgecolor='#34495e'))
        ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=9)
    def link(a,b):
        x,y,w,h,_=boxes[a];xx,yy,ww,hh,_=boxes[b]
        ax.annotate('',xy=(xx,yy+hh/2),xytext=(x+w,y+h/2),arrowprops=dict(arrowstyle='->',color='#34495e'))
    for a,b in [('sequence','encoder'),('station','encoder'),('encoder','expert0'),('encoder','gate'),('encoder','expert1'),('expert0','output'),('gate','output'),('expert1','output')]:link(a,b)
    ax.text(5.5,.3,'Experts include dropout 0.2; mixture is formed in standardized log-target space.',ha='center',fontsize=9)
    fig.tight_layout();fig.savefig(D['out']/'moe_architecture.png',dpi=300);plt.close(fig)

plot_architecture(D)

## Step 9 — Check model execution before long training

Tests a forward and backward pass for every distinct architecture with tiny synthetic arrays. These are implementation checks only; synthetic arrays do not enter the research dataset.

In [ ]:
def check_models():
    import tensorflow as tf
    seen=set()
    for spec in experiment_specs('full'):
        signature=(spec['k'],spec['embed'],spec['gate'],spec['kind'],spec['width'])
        if signature in seen: continue
        seen.add(signature);tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(100)
        model=build_model(spec,2)
        x={'seq_in':np.random.default_rng(2).normal(size=(2,L,len(FEATURES))).astype('float32')}
        if spec['embed']: x['station_in']=np.array([0,1],dtype='int32')
        y=np.zeros((2,H),'float32')
        model.compile(optimizer=tf.keras.optimizers.Adam(LR),loss=loss_for(spec,4.,1.))
        loss=model.train_on_batch(x,y)
        pred=model(x,training=False).numpy()
        expected=(2,H,2) if spec['kind']=='prob' else (2,H)
        assert pred.shape==expected and np.isfinite(pred).all() and np.isfinite(loss)
        print('Model execution check passed:',spec['name'])
    tf.keras.backend.clear_session()

if RUN_MODEL_CHECKS: check_models()

## Step 10 — Train the full study and save comparable predictions

Training runs sequentially through the chosen configurations and seeds. Each completed experiment writes model weights, architecture/settings, learning history, predictions, per-horizon metrics, station metrics and runtime. Keep the default three seeds for the final comparison. Use validation metrics to assess configurations; do not choose the best test seed.

**Metric consistency:** pooled RMSE is $\sqrt{\operatorname{mean}(y-\hat y)^2}$ over all target positions. Mean horizon RMSE is exported separately and must not be mixed with pooled RMSE. $R^2=1-\sum(y-\hat y)^2/\sum(y-\bar y)^2$; subset R² uses the subset mean. Negative subset R² means worse squared error than that retrospective constant reference.

**Back-transformation:** deterministic forecasts use `expm1` as the primary result. With weighted absolute-error training this is not automatically an arithmetic conditional mean. Training-residual smearing is an exploratory sensitivity, not a guarantee of unbiasedness; it uses no test residuals. The probabilistic model exports both lognormal median and mean, 80% coverage, coverage error, interval width and interval score. No correction is chosen based on test scores.

**Baseline definitions:** persistence repeats the last observed PM2.5; seasonal naive copies the last 24 observed input values into the next 24 hours. Both use exactly the same eligible test windows.

In [ ]:
def metrics(y,p):
    y=np.asarray(y,dtype='float64');p=np.asarray(p,dtype='float64')
    if y.shape!=p.shape or not np.isfinite(p).all(): raise ValueError('Invalid prediction shape or nonfinite values.')
    ans={}
    for label,mask in [('overall',np.ones(y.shape,bool)),('high',y>=THRESHOLD)]:
        a=y[mask];b=p[mask];den=np.sum((a-a.mean())**2) if len(a) else 0
        ans.update({label+'_n':len(a),label+'_MAE':np.mean(abs(a-b)) if len(a) else np.nan,
                    label+'_RMSE':np.sqrt(np.mean((a-b)**2)) if len(a) else np.nan,
                    label+'_R2':1-np.sum((a-b)**2)/den if den>0 else np.nan,
                    label+'_bias':np.mean(b-a) if len(a) else np.nan})
    if y.ndim==2: ans['mean_horizon_RMSE_secondary']=np.sqrt(np.mean((y-p)**2,axis=0)).mean()
    return ans

def inverse(z,D):
    with np.errstate(over='raise',invalid='raise'):
        return np.maximum(np.expm1(np.asarray(z,dtype='float64')*D['ys']+D['ym']),0).astype('float32')

def dataset(D,part,spec,training=False,seed=1,limit=None):
    import tensorflow as tf
    p=D['data'][part]; ids=np.arange(len(p['y']))
    if limit is not None: ids=np.random.default_rng(seed).permutation(ids)[:limit]
    ds=tf.data.Dataset.from_tensor_slices(ids)
    if training: ds=ds.shuffle(min(len(ids),20000),seed=seed,reshuffle_each_iteration=True)
    x=tf.convert_to_tensor(D['x']); origins=tf.convert_to_tensor(p['origin']);
    sid=tf.convert_to_tensor(p['sid']); targets=tf.convert_to_tensor(p['z'])
    def batch(i):
        o=tf.gather(origins,i); inp={'seq_in':tf.gather(x,o[:,None]+tf.range(-L+1,1,dtype=tf.int64))}
        if spec['embed']: inp['station_in']=tf.gather(sid,i)
        return inp,tf.gather(targets,i)
    return ds.batch(BATCH).map(batch,num_parallel_calls=1).prefetch(1)

def predict_split(model,D,part,spec):
    return model.predict(dataset(D,part,spec).map(lambda x,y:x),verbose=0)

def interval_metrics(y,lo,hi,alpha=.2):
    covered=(y>=lo)&(y<=hi); width=hi-lo
    score=width+2/alpha*(lo-y)*(y<lo)+2/alpha*(y-hi)*(y>hi)
    return dict(nominal_coverage=1-alpha,PICP=float(covered.mean()),
                coverage_error=float(covered.mean()-(1-alpha)),
                mean_width=float(width.mean()),interval_score=float(score.mean()))

def run_one(D,spec,seed,smoke=False):
    import tensorflow as tf
    run=D['out']/f"{spec['name']}_seed{seed}";run.mkdir(exist_ok=True)
    complete=run/'complete.json'
    if complete.exists():
        print('Reusing completed',run.name);return json.loads(complete.read_text())
    tf.keras.backend.clear_session();gc.collect();tf.keras.utils.set_random_seed(seed)
    model=build_model(spec,len(D['stations']))
    model.compile(optimizer=tf.keras.optimizers.Adam(LR),loss=loss_for(spec,D['ym'],D['ys']))
    save_json(run/'architecture.json',json.loads(model.to_json()))
    save_json(run/'settings.json',dict(spec=spec,seed=seed,parameters=model.count_params(),
              hidden=HIDDEN,embedding=EMB_DIM,dropout=DROPOUT,epochs=EPOCHS,batch=BATCH,lr=LR))
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=PATIENCE,restore_best_weights=True),
               tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',patience=2,factor=.5,min_lr=1e-5),
               tf.keras.callbacks.ModelCheckpoint(str(run/'best.weights.h5'),save_weights_only=True,save_best_only=True)]
    started=time.perf_counter()
    if spec['kind']=='tl_lstm_mha':
        # Both phases use only the main training partition. Validation remains disjoint.
        p=D['data']['train'];origin=pd.to_datetime(p['time_ns'],utc=True).tz_convert(TZ)
        phaseD=dict(D);phaseD['data']=dict(D['data'])
        for year,label in [(2022,'pretrain_2022'),(2023,'finetune_2023')]:
            mask=((origin-pd.Timedelta(hours=L-1)).year==year)&((origin+pd.Timedelta(hours=H)).year==year)
            if not mask.any():raise ValueError(f'Transfer baseline requires training windows in {year}.')
            phaseD['data'][label]={k:v[mask] for k,v in p.items()}
        model.compile(optimizer=tf.keras.optimizers.Nadam(TL_LR),loss=loss_for(spec,D['ym'],D['ys']))
        pre_callbacks=[tf.keras.callbacks.EarlyStopping(patience=PATIENCE,restore_best_weights=True),
                       tf.keras.callbacks.ReduceLROnPlateau(patience=2,factor=.5,min_lr=1e-5)]
        pre=model.fit(dataset(phaseD,'pretrain_2022',spec,True,seed,512 if smoke else None),
                      validation_data=dataset(D,'val',spec,limit=256 if smoke else None),
                      epochs=1 if smoke else EPOCHS,callbacks=pre_callbacks,verbose=2)
        pd.DataFrame(pre.history).to_csv(run/'pretraining_history.csv',index_label='epoch_zero_based')
        model.save_weights(run/'pretrained_2022.weights.h5')
        model.get_layer('transfer_lstm').trainable=False
        model.compile(optimizer=tf.keras.optimizers.Nadam(TL_LR),loss=loss_for(spec,D['ym'],D['ys']))
        history=model.fit(dataset(phaseD,'finetune_2023',spec,True,seed,512 if smoke else None),
                      validation_data=dataset(D,'val',spec,limit=256 if smoke else None),
                      epochs=1 if smoke else EPOCHS,callbacks=callbacks,verbose=2)
        save_json(run/'transfer_protocol.json',dict(pretrain_year=2022,finetune_year=2023,
                  frozen_layer='transfer_lstm',optimizer='Nadam',lr=TL_LR,
                  pretrain_epochs=len(pre.history['loss']),finetune_epochs=len(history.history['loss']),
                  validation='common disjoint validation partition for both phases'))
    else:
        history=model.fit(dataset(D,'train',spec,True,seed,512 if smoke else None),
                      validation_data=dataset(D,'val',spec,limit=256 if smoke else None),
                      epochs=1 if smoke else EPOCHS,callbacks=callbacks,verbose=2)
    elapsed=time.perf_counter()-started
    model.load_weights(run/'best.weights.h5')
    pd.DataFrame(history.history).to_csv(run/'training_history.csv',index_label='epoch_zero_based')
    predictions={}
    for part in ['val','test']:
        z=predict_split(model,D,part,spec);predictions[part]=z
        if spec['kind']=='prob':
            mu=z[...,0]*D['ys']+D['ym'];sig=np.logaddexp(0,z[...,1])*D['ys']+1e-6
            p=np.maximum(np.expm1(mu),0).astype('float32')  # conditional median: matches MAE point estimand
            mean=np.maximum(np.expm1(mu+.5*sig**2),0)
            if not np.isfinite(mean).all(): raise FloatingPointError('Nonfinite lognormal mean.')
            np.save(run/f'{part}_lognormal_mean.npy',mean)
            if part=='test':
                y=D['data'][part]['y'];lo=np.maximum(np.expm1(mu-norm.ppf(.9)*sig),0);hi=np.maximum(np.expm1(mu+norm.ppf(.9)*sig),0)
                rows=[dict(horizon='all',**interval_metrics(y,lo,hi))]
                rows += [dict(horizon=h+1,**interval_metrics(y[:,h],lo[:,h],hi[:,h])) for h in range(H)]
                m=y>=THRESHOLD
                if m.any(): rows.append(dict(horizon='high_subset',**interval_metrics(y[m],lo[m],hi[m])))
                pd.DataFrame(rows).to_csv(run/'interval_calibration.csv',index=False)
                save_json(run/'lognormal_mean_metrics.json',metrics(y,mean))
                np.savez_compressed(run/'intervals.npz',lower=lo,upper=hi)
        else: p=inverse(z,D)
        np.save(run/f'{part}_prediction.npy',p)
    # Train-residual smearing is a sensitivity analysis, not a claim of unbiased means under weighted MAE.
    if spec['kind']!='prob':
        ds=dataset(D,'train',spec,limit=20000,seed=seed)
        residuals=[]
        for inp,yt in ds:
            zp=model(inp,training=False).numpy()
            residuals.append((yt.numpy()-zp)*D['ys'])
        r=np.concatenate(residuals).astype('float64')
        factor=np.mean(np.exp(r),axis=0)
        save_json(run/'training_smearing_factors.json',factor.tolist())
        for part in ['val','test']:
            z=predictions[part]
            corrected=np.maximum(np.exp(z.astype('float64')*D['ys']+D['ym'])*factor-1,0)
            pd.DataFrame([dict(method='plain_expm1',**metrics(D['data'][part]['y'],np.load(run/f'{part}_prediction.npy'))),
                          dict(method='training_residual_smearing',**metrics(D['data'][part]['y'],corrected))]).to_csv(run/f'{part}_backtransform_sensitivity.csv',index=False)
    y=D['data']['test']['y'];p=np.load(run/'test_prediction.npy')
    pd.DataFrame([dict(horizon=h+1,**metrics(y[:,h],p[:,h])) for h in range(H)]).to_csv(run/'horizon_metrics.csv',index=False)
    pd.DataFrame([dict(station_id=D['stations'][s],**metrics(y[m],p[m])) for s in np.unique(D['data']['test']['sid'])
                  for m in [D['data']['test']['sid']==s]]).to_csv(run/'station_metrics.csv',index=False)
    if spec['k']>1 and spec['gate']=='learned':
        inspector=tf.keras.Model(model.inputs,[model.get_layer('gate').output]+[model.get_layer(f'expert_{i}').output for i in range(spec['k'])])
        values=inspector.predict(dataset(D,'test',spec).map(lambda x,y:x),verbose=0)
        gates=values[0]
        if spec['k']==2: gates=np.concatenate([1-gates,gates],axis=1)
        pd.DataFrame({**{f'expert_{i}_weight':gates[:,i] for i in range(spec['k'])},
                     'future_max_observed':y.max(axis=1),'future_high_window':(y>=THRESHOLD).any(axis=1)}).to_csv(run/'gate_diagnostics.csv',index=False)
        pd.DataFrame([dict(expert=i,**metrics(y,inverse(z,D))) for i,z in enumerate(values[1:])]).to_csv(run/'expert_branch_metrics.csv',index=False)
    row=dict(model=spec['name'],seed=seed,training_seconds=elapsed,epochs_run=len(history.history['loss']),
             parameters=model.count_params(),validation_RMSE=metrics(D['data']['val']['y'],np.load(run/'val_prediction.npy'))['overall_RMSE'],
             smoke_only=smoke,**metrics(y,p))
    save_json(run/'architecture_after_training.json',json.loads(model.to_json()))
    save_json(complete,row);del model;gc.collect();return row

def evaluate_suite(D,specs,seeds,smoke=False):
    rows=[]
    for spec in specs:
        for seed in seeds: rows.append(run_one(D,spec,seed,smoke))
    test=D['data']['test'];y=test['y'];o=test['origin']
    heuristics={'persistence':np.repeat(D['raw_x'][o,0,None],H,axis=1),
                'seasonal_naive':D['raw_x'][o[:,None]+np.arange(-H+1,1),0]}
    for name,p in heuristics.items():
        run=D['out']/name;run.mkdir(exist_ok=True);np.save(run/'test_prediction.npy',p)
        rows.append(dict(model=name,seed=-1,smoke_only=smoke,**metrics(y,p)))
    table=pd.DataFrame(rows);table.to_csv(D['out']/'all_runs.csv',index=False)
    metrics_cols=['overall_MAE','overall_RMSE','overall_R2','high_MAE','high_RMSE','high_R2','training_seconds','parameters']
    table.groupby('model')[metrics_cols].agg(['mean','std','count']).to_csv(D['out']/'seed_summary.csv')
    # Rankings for sensitivity selection use validation only; test values never select a run.
    table[table.seed>=0].groupby('model').validation_RMSE.mean().sort_values().to_csv(D['out']/'validation_ranking.csv')
    if smoke: print('SMOKE CHECK ONLY: these estimates cannot be used in the paper.')
    return table

RESULTS = evaluate_suite(D,SPECS,ACTIVE_SEEDS,smoke=MODE=='smoke')
print(RESULTS[['model','seed','overall_RMSE','high_RMSE','high_R2']].to_string(index=False))

## Step 11 — Generate all result figures and manuscript tables

Produces the observed/predicted panels with visible 1:1 lines, horizon-error curves, residual histograms/scatter, training curves, gate boxplots and intensity trend, descriptive gate correlations, model comparison, seed means/SDs, runtime/parameter tables and a reviewer coverage checklist.

The figure seed is prespecified, not selected for its test performance. Gate correlations with future observed concentrations are retrospective descriptive evidence; ordinary iid correlation p-values are deliberately omitted. Interpretability alone does not establish that gating caused an improvement.

If coordinates are unavailable, this step writes `station_coordinates_TO_COMPLETE.csv`. Enter verified coordinates, set `STATION_METADATA` in Step 2, and rerun this step to generate the station-location figure.

In [ ]:
def export_figures(D,seed):
    folder=D['out']/f'moe2_w3_seed{seed}'
    y=D['data']['test']['y'];p=np.load(folder/'test_prediction.npy')
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    rng=np.random.default_rng(100)
    for ax,mask,title in zip(axes,[np.ones(y.shape,bool),y>=THRESHOLD],['All targets','High pollution targets']):
        yy=y[mask];pp=p[mask]
        if not len(yy): ax.set_title(title+' (empty)');continue
        ids=rng.choice(len(yy),min(10000,len(yy)),replace=False)
        ax.scatter(yy[ids],pp[ids],s=3,alpha=.2)
        lo=min(0,float(pp.min()));hi=max(float(yy.max()),float(pp.max()))
        ax.plot([lo,hi],[lo,hi],'k--',lw=1.5,zorder=10,label='1:1')
        ax.set(xlim=(lo,hi),ylim=(lo,hi),xlabel='Observed PM2.5 (µg/m³)',ylabel='Predicted PM2.5 (µg/m³)',title=title)
        ax.set_aspect('equal',adjustable='box');ax.legend()
    fig.tight_layout();fig.savefig(D['out']/'observed_predicted.png',dpi=300);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    for ax,err,title in [(axes[0],np.mean(abs(y-p),axis=0),'MAE'),(axes[1],np.sqrt(np.mean((y-p)**2,axis=0)),'RMSE')]:
        ax.plot(np.arange(1,H+1),err);ax.set(xlabel='Forecast lead hour',ylabel=title+' (µg/m³)')
    fig.tight_layout();fig.savefig(D['out']/'horizon_errors.png',dpi=300);plt.close(fig)
    if STATION_METADATA is None:
        pd.DataFrame({'station_id':D['stations'],'latitude':np.nan,'longitude':np.nan}).to_csv(D['out']/'station_coordinates_TO_COMPLETE.csv',index=False)
        print('Coordinates unavailable: complete station_coordinates_TO_COMPLETE.csv with verified locations, set STATION_METADATA, rerun this cell.')
    else:
        coords=pd.read_csv(STATION_METADATA,dtype={'station_id':str})
        if coords.station_id.duplicated().any(): raise ValueError('Duplicate station coordinate IDs.')
        coords=pd.DataFrame({'station_id':D['stations']}).merge(coords,on='station_id',how='left',validate='one_to_one')
        if coords[['latitude','longitude']].isna().any().any(): raise ValueError('Coordinates incomplete.')
        if not(coords.latitude.between(-90,90)&coords.longitude.between(-180,180)).all(): raise ValueError('Invalid coordinates.')
        coords.to_csv(D['out']/'station_coordinates.csv',index=False)
        fig,ax=plt.subplots(figsize=(8,7));ax.scatter(coords.longitude,coords.latitude)
        for r in coords.itertuples():ax.annotate(r.station_id,(r.longitude,r.latitude),fontsize=6)
        ax.set(xlabel='Longitude',ylabel='Latitude',title='Monitoring station locations')
        ax.set_aspect(1/np.cos(np.deg2rad(coords.latitude.mean())))
        fig.tight_layout();fig.savefig(D['out']/'station_location_map.png',dpi=300);plt.close(fig)
    checklist=[
      ('R1.5 R4.9','two versus three experts','full suite: moe3_w3; validation ranking'),
      ('R1.6 R4.7','loss weight sensitivity','full suite: weights 1,2,3,4,5,10'),
      ('R2.1 R4.6','separate architecture and loss','single_w1 single_w3 moe2_w1 moe2_w3; fixed gate; no station'),
      ('R1.7 R2.9 R4.4 R4.12 R4.20','variability and dependence','seed_summary, daily HAC DM, global Holm, calendar block bootstrap'),
      ('R2.3 R2.5 R3.4 R3.5','data and missingness','data_audit, station_coverage, missingness tables, manifests'),
      ('R2.4','station map','requires verified coordinates from authors'),
      ('R2.6 R4.19','back transformation','training residual smearing sensitivity; lognormal mean/median'),
      ('R2.7','seasonal naive','copies last 24 observed input hours'),
      ('R3.2 R3.3 R3.6 R4.8','reproducibility','per-target weighted MAE; preprocessing.json; architecture/settings'),
      ('R4.14','probabilistic calibration','PICP, coverage error, width and interval score by horizon'),
      ('R4.16 R4.17','runtime and figures','training_seconds; observed_predicted.png'),
      ('Editorial R1.1 R1.2','title affiliations declarations','MANUSCRIPT EDIT REQUIRED'),
      ('R1.3 R1.4 R1.8 R2.2 R2.8 R2.10 R2.11 R3.1 R3.7 R3.8 R3.9 R3.10 R4.1 R4.2 R4.3 R4.5 R4.10 R4.11 R4.13 R4.15 R4.18 R4.21',
       'literature novelty limits metric interpretation and code availability','AUTHOR WRITING AND SOURCE VERIFICATION REQUIRED')]
    pd.DataFrame(checklist,columns=['comments','topic','evidence_or_remaining_action']).to_csv(D['out']/'reviewer_coverage.csv',index=False)

def plot_result_diagnostics(D,table,seed):
    from scipy.stats import spearmanr
    out=D['out'];run=out/f'moe2_w3_seed{seed}';y=D['data']['test']['y'];p=np.load(run/'test_prediction.npy')
    residual=(p-y).ravel();observed=y.ravel();rng=np.random.default_rng(1)
    ix=rng.choice(len(residual),min(20000,len(residual)),replace=False)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].hist(residual,bins=80);axes[0].axvline(0,color='black',ls='--')
    axes[0].set(xlabel='Predicted − observed (µg/m³)',ylabel='Count',title=f'Mean residual {residual.mean():.2f}')
    axes[1].scatter(observed[ix],residual[ix],s=3,alpha=.15);axes[1].axhline(0,color='black',ls='--')
    axes[1].set(xlabel='Observed PM2.5 (µg/m³)',ylabel='Residual (µg/m³)',title='Residual versus observed concentration')
    fig.tight_layout();fig.savefig(out/'residual_diagnostics.png',dpi=300);plt.close(fig)
    hist=pd.read_csv(run/'training_history.csv')
    fig,ax=plt.subplots(figsize=(7,4));ax.plot(np.arange(1,len(hist)+1),hist.loss,label='Training')
    ax.plot(np.arange(1,len(hist)+1),hist.val_loss,label='Validation');ax.legend()
    ax.set(xlabel='Epoch',ylabel='Weighted MAE in standardized log space',title=f'MoE reference learning curves seed {seed}')
    fig.tight_layout();fig.savefig(out/'reference_learning_curves.png',dpi=300);plt.close(fig)
    gate=pd.read_csv(run/'gate_diagnostics.csv');groups=gate.future_high_window.astype(bool)
    vals=gate.expert_1_weight
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    samples=[vals[~groups].to_numpy(),vals[groups].to_numpy()]
    axes[0].boxplot(samples,showfliers=False)
    axes[0].set_xticks([1,2],['No high target','At least one high target'])
    axes[0].set(ylabel='Expert 1 gate weight',title='Retrospective window grouping')
    axes[1].scatter(gate.future_max_observed,vals,s=3,alpha=.08)
    bins=pd.qcut(gate.future_max_observed,q=20,duplicates='drop')
    b=gate.assign(bin=bins).groupby('bin',observed=True).agg(peak=('future_max_observed','mean'),weight=('expert_1_weight','mean'))
    axes[1].plot(b.peak,b.weight,color='red');axes[1].axvline(THRESHOLD,color='black',ls='--')
    axes[1].set(xlabel='Future observed 24-hour maximum (µg/m³)',ylabel='Expert 1 gate weight')
    fig.tight_layout();fig.savefig(out/'gate_interpretability.png',dpi=300);plt.close(fig)
    corr=gate[['expert_1_weight','future_max_observed']].copy();corr['future_mean_observed']=y.mean(axis=1)
    corr.corr(method='spearman').to_csv(out/'gate_spearman_descriptive.csv')
    # Ordinary correlation p-values would ignore overlapping-window dependence; report coefficients only.
    pd.DataFrame([dict(group='high' if label else 'non_high',n=int((groups==label).sum()),
                        mean=float(vals[groups==label].mean())) for label in [False,True]]).to_csv(out/'gate_group_summary.csv',index=False)
    fig,ax=plt.subplots(figsize=(6,5));matrix=corr.corr(method='spearman')
    im=ax.imshow(matrix,vmin=-1,vmax=1,cmap='coolwarm')
    ax.set_xticks(range(3),matrix.columns,rotation=35,ha='right');ax.set_yticks(range(3),matrix.columns)
    for i in range(3):
        for j in range(3):ax.text(j,i,f'{matrix.iloc[i,j]:.2f}',ha='center',va='center')
    fig.colorbar(im,ax=ax);fig.tight_layout();fig.savefig(out/'gate_correlation_heatmap.png',dpi=300);plt.close(fig)
    # Model comparison: direct common-metric table, never hard-coded results.
    names=['overall_MAE','overall_RMSE','overall_R2','high_MAE','high_RMSE','high_R2']
    means=table.groupby('model')[names].mean();stds=table.groupby('model')[names].std()
    means.to_csv(out/'comparison_means.csv');stds.to_csv(out/'comparison_seed_sd.csv')
    order=means.sort_values('overall_RMSE').index
    fig,ax=plt.subplots(figsize=(10,max(4,len(order)*.3)))
    ax.barh(order,means.loc[order,'overall_RMSE'],xerr=stds.loc[order,'overall_RMSE'].fillna(0))
    ax.invert_yaxis();ax.set(xlabel='Pooled RMSE (µg/m³); error bars: seed SD',title='Common test set model comparison')
    fig.tight_layout();fig.savefig(out/'model_comparison.png',dpi=300);plt.close(fig)
    ref=means.loc['moe2_w3','overall_RMSE']
    improvement=pd.DataFrame({'baseline_RMSE':means.overall_RMSE,'reference_RMSE':ref})
    improvement['absolute_reduction']=improvement.baseline_RMSE-ref
    improvement['relative_reduction_pct']=100*improvement.absolute_reduction/improvement.baseline_RMSE
    improvement.to_csv(out/'rmse_improvement_from_seed_means.csv')
    configs=pd.read_csv(out/'experiment_plan.csv')
    configs=configs.merge(table.groupby('model')[['parameters','training_seconds','epochs_run']].mean(),left_on='name',right_index=True,how='left')
    configs.to_csv(out/'architecture_and_runtime_table.csv',index=False)
    print('Diagnostic figures and comparison tables saved in',out)

export_figures(D,ACTIVE_SEEDS[0])
plot_result_diagnostics(D,RESULTS,ACTIVE_SEEDS[0])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 12 — Run confidence intervals and corrected paired tests

**Reviewer issues addressed: baseline variability, confidence intervals, overlapping forecast errors and multiple testing.**

The test compares daily average squared losses per horizon, pooling stations/origins within each day. It uses a 7-day HAC long-run variance: an approximate daily-aggregated DM-style comparison. Calendar spacing is retained across missing campaign days. Two-sided p-values receive one global Holm correction across all baselines, horizons and seeds; a corrected p below 0.05 plus a positive baseline-minus-reference difference counts as a reference win.

Paired calendar moving-block bootstrap resamples all stations jointly using 3-, 7- and 14-day blocks. It reports pointwise 95% percentile intervals for absolute metrics and paired improvements. These intervals are not simultaneous multiple-testing intervals. Inspect block-length sensitivity and the plausibility of dependence assumptions. Seed variation and test-sample uncertainty are distinct; three seeds are not three independent test datasets.

2024 already informed the original study. Describe it as the fixed evaluation set and disclose revision analyses, rather than calling it a newly untouched test set.

In [ ]:
def holm_adjust(p):
    p=np.asarray(p,float);order=np.argsort(p);n=len(p)
    adjusted=np.empty(n);adjusted[order]=np.minimum(1,np.maximum.accumulate(p[order]*(n-np.arange(n))))
    return adjusted

def daily_losses(D,y,p):
    dates=pd.to_datetime(D['data']['test']['time_ns'],utc=True).tz_convert(TZ).normalize()
    sq=pd.DataFrame((np.asarray(y,float)-p)**2,index=dates)
    return sq.groupby(level=0).mean().asfreq('D')

def hac_dm(a,b,lag=7):
    # Daily loss differential baseline minus reference; all stations aggregated jointly per day.
    # Calendar grid preserves actual lag spacing through campaign gaps.
    d=np.asarray(a-b,float);ok=np.isfinite(d);n=int(ok.sum())
    if n<max(20,lag+3): return dict(statistic=np.nan,p=np.nan,n_days=n,mean_difference=np.nan)
    mean=d[ok].mean();center=np.where(ok,d-mean,0.)
    lrv=np.dot(center,center)/n
    for k in range(1,min(lag,len(d)-1)+1):
        lrv+=2*(1-k/(lag+1))*np.dot(center[k:],center[:-k])/n
    stat=mean/np.sqrt(lrv/n) if lrv>0 else np.nan
    return dict(statistic=stat,p=2*norm.sf(abs(stat)) if np.isfinite(stat) else np.nan,
                n_days=n,mean_difference=mean)

def bootstrap_comparison(D,y,reference,baseline,block_days=7,n_boot=1000):
    # Sufficient statistics keep bootstrap cost independent of the number of windows.
    dates=pd.to_datetime(D['data']['test']['time_ns'],utc=True).tz_convert(TZ).normalize()
    rows=[]
    for label,mask in [('overall',np.ones(y.shape,bool)),('high',y>=THRESHOLD)]:
        yy=np.asarray(y,float);r=np.asarray(reference,float);b=np.asarray(baseline,float)
        frame=pd.DataFrame(dict(n=mask.sum(1),sy=(yy*mask).sum(1),sy2=(yy**2*mask).sum(1),
                    rsse=((yy-r)**2*mask).sum(1),bsse=((yy-b)**2*mask).sum(1),
                    rae=(abs(yy-r)*mask).sum(1),bae=(abs(yy-b)*mask).sum(1)),index=dates)
        daily=frame.groupby(level=0).sum().asfreq('D',fill_value=0).to_numpy()
        nd=len(daily)
        if nd<2*block_days: continue
        rng=np.random.default_rng(2026);stats=[]
        for _ in range(n_boot):
            starts=rng.integers(0,nd-block_days+1,size=int(np.ceil(nd/block_days)))
            ix=(starts[:,None]+np.arange(block_days)).ravel()[:nd]
            n,sy,sy2,rs,bs,ra,ba=daily[ix].sum(0)
            if n<=0: continue
            den=sy2-sy**2/n
            stats.append([np.sqrt(rs/n),np.sqrt(bs/n),np.sqrt(bs/n)-np.sqrt(rs/n),
                          ra/n,ba/n,(ba-ra)/n,1-rs/den if den>0 else np.nan,
                          1-bs/den if den>0 else np.nan])
        if not stats: continue
        names=['reference_RMSE','baseline_RMSE','RMSE_improvement','reference_MAE',
               'baseline_MAE','MAE_improvement','reference_R2','baseline_R2']
        for j,name in enumerate(names):
            vals=np.asarray(stats)[:,j];vals=vals[np.isfinite(vals)]
            if len(vals):rows.append(dict(subset=label,metric=name,block_days=block_days,
                bootstrap_replicates=len(vals),ci_lower=np.quantile(vals,.025),ci_upper=np.quantile(vals,.975)))
    return rows

def paired_comparisons(D,specs,seeds):
    y=D['data']['test']['y']
    heuristics=['persistence','seasonal_naive']
    # Paired tests compare identically indexed forecast cases, one test set per random seed.
    dm=[];cis=[]
    for seed in seeds:
        ref=np.load(D['out']/f'moe2_w3_seed{seed}'/'test_prediction.npy')
        refdaily=daily_losses(D,y,ref)
        for name in [s['name'] for s in specs if s['name']!='moe2_w3']+list(heuristics):
            path=D['out']/(name if name in heuristics else f'{name}_seed{seed}')/'test_prediction.npy'
            p=np.load(path);bdaily=daily_losses(D,y,p)
            for h in range(H):
                dm.append(dict(baseline=name,seed=seed,horizon=h+1,hac_lag_days=7,
                               **hac_dm(bdaily.iloc[:,h],refdaily.iloc[:,h],7)))
            for block in BLOCK_DAYS:
                cis += [dict(baseline=name,seed=seed,**row) for row in bootstrap_comparison(D,y,ref,p,block,BOOTSTRAPS)]
    dm=pd.DataFrame(dm);dm['p_holm_global']=np.nan
    valid=dm.p.notna();dm.loc[valid,'p_holm_global']=holm_adjust(dm.loc[valid,'p'])
    dm['reference_win']=(dm.mean_difference>0)&(dm.p_holm_global<.05)
    dm.to_csv(D['out']/'daily_HAC_DM_global_Holm.csv',index=False)
    dm.groupby(['baseline','seed']).reference_win.sum().to_csv(D['out']/'DM_corrected_win_counts.csv')
    pd.DataFrame(cis).to_csv(D['out']/'paired_calendar_block_bootstrap_CI.csv',index=False)
    return dm

if MODE!='smoke':
    DM_RESULTS=paired_comparisons(D,SPECS,ACTIVE_SEEDS)
    print(DM_RESULTS.groupby(['baseline','seed']).reference_win.sum())
else:
    print('Statistical inference skipped for smoke-only results.')

## Step 13 — Optionally examine shorter forward-fill limits

The main 6-hour policy is already evaluated. Set `RUN_FILL_SENSITIVITY=True` in Step 2 to add 0-, 1- and 3-hour policies for the reference model. These add nine experiments with three seeds. The alternative policies refit training-only preprocessing and preserve observed-target window eligibility.

In [ ]:
if RUN_FILL_SENSITIVITY:
    for hours in [0,1,3]:
        alt=prepare_data(DATA_PATH,D['out']/f'fill_sensitivity_{hours}h',hours)
        for seed in ACTIVE_SEEDS:
            run_one(alt,experiment_specs('core')[0],seed,smoke=MODE=='smoke')
else:
    print('Optional fill sensitivity disabled; main 6-hour policy complete.')

## Step 14 — Package the results

Creates a ZIP containing CSV/JSON reports and figures. Large prediction arrays and weights remain in the results folder. Share the ZIP for the manuscript revision and point-by-point response.

In [ ]:
def package_outputs(D):
    import zipfile
    # Exclude large model weights, predictions and extracted raw files. Those remain in Drive.
    archive=D['out']/'reviewer_results_and_figures.zip'
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for path in D['out'].rglob('*'):
            if path.is_file() and path.suffix.lower() in ['.csv','.json','.png','.txt']:
                z.write(path,path.relative_to(D['out']))
    print('Summary package:',archive)
    return archive

RESULT_ARCHIVE = package_outputs(D)

## What remains outside code

The generated `reviewer_coverage.csv` maps outputs to the reviewer comments. The following require scientific or editorial writing after results are available:

- Revise title and institution superscripts; separate Ethical approval, Consent to participate and Consent to publish.
- Verify the dataset citation/version, station locations, coverage period and threshold attribution.
- Update related work and clarify novelty relative to your earlier publications. Reviewer-suggested references should be included only when relevant, as the editor states.
- Discuss crop-burning changes, omitted meteorology/satellite predictors, scalar gating, modest R², spatial dependence and untested transfer to other cities. Observed distribution shifts do not prove their cause.
- Explain the revised preprocessing, any changed sample counts, updated baseline target space and regenerated results.
- Publish code/data only where permissions permit and supply the final repository URL in the manuscript.

Use **seed means and SDs** consistently in the final tables and state clearly when a figure shows one prespecified seed. Do not claim a statistically significant gain unless the generated analysis supports it.

Implementation references: [Keras training API](https://keras.io/api/models/model_training_apis/) and [Keras Dot layer](https://keras.io/api/layers/merging_layers/dot/). Verify and cite the appropriate methodological sources for HAC/DM, multiple testing and block bootstrap in the manuscript.